# Estudo Pandas: Pipeline de Vendas

**Objetivo:** Limpeza e inspeção de base de vendas tech + cruzamento com cadastro de gerentes.  
**Autor:** Gu  
**Data:** 2026-06-01

---

## 1. Setup

Bibliotecas necessárias para ETL e visualização.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configurações de display para facilitar inspeção
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## 2. Dicionário de Dados

| Base | Coluna | Tipo Esperado | Descrição |
|------|--------|---------------|-----------|
| vendas | ID_Pedido | int | Identificador único do pedido |
| vendas | Data | datetime | Data da transação |
| vendas | Loja | str | Cidade da loja (pode conter inconsistências de case) |
| vendas | Produto | str | Nome do produto comercializado |
| vendas | Preco_Unitario | float | Valor unitário em R$ |
| vendas | Qtd | int | Quantidade vendida |
| vendas | Cliente | str | Código anonimizado do cliente |
| vendas | Data_Base | datetime | Data de snapshot da base (quase 100% nulo) |
| gerentes | Loja | str | Cidade da loja (chave para join) |
| gerentes | Gerente | str | Nome do responsável |
| gerentes | Meta_Mensal | int | Meta de faturamento em R$ |

## 3. Data Loading

**Decisões:**
- `low_memory=False` força inferência de tipos em uma única passada (base > 100k linhas).
- `parse_dates` já converte colunas de data no carregamento, evitando conversão posterior.

In [ ]:
df_vendas = pd.read_csv(
    'vendas_tech.csv',
    sep=',',
    low_memory=False,
    parse_dates=['Data', 'Data_Base']  # conversão imediata de datas
)

df_gerentes = pd.read_excel('gerentes_lojas.xlsx')

# Sanity check: schema e amostra
print(f'Vendas: {df_vendas.shape[0]} rows, {df_vendas.shape[1]} cols')
print(f'Gerentes: {df_gerentes.shape[0]} rows, {df_gerentes.shape[1]} cols')

## 4. Inspeção Inicial (EDA Rápida)

Aqui não se explica o que `head()` faz. A célula é autoexplicativa pelo título da seção.

In [ ]:
df_vendas.head()

In [ ]:
df_vendas.tail()

In [ ]:
df_gerentes

### 4.1 Schema e Qualidade

`info()` + `describe()` para detectar nulos, tipos incorretos e outliers de primeira ordem.

In [ ]:
df_vendas.info()

In [ ]:
df_vendas.describe()

## 5. ETL — Limpeza

**Pipeline:**
1. Padronizar case da coluna `Loja` (RIO DE JANEIRO → Rio de Janeiro).
2. Dropar `Data_Base` (1 non-null em 100k linhas = sem valor analítico).
3. Tratar nulos em `Loja` (2% missing).
df_vendas.info()

In [ ]:
# Cópia explícita para não mutar o original durante experimentos
df = df_vendas.copy()

# 1. Normalização de texto
df['Loja'] = df['Loja'].str.title()

# 2. Drop de coluna com cardinalidade/missing inviável
df = df.drop(columns=['Data_Base'])

# 3. Tratamento de nulos: como Loja é dimensão analítica, dropamos linhas sem loja
# (alternativa: imputar 'Desconhecido', mas isso quebraria o join com gerentes)
df = df.dropna(subset=['Loja'])

# 4. Duplicatas (considerando todas as colunas como chave lógica)
duplicatas = df.duplicated().sum()
print(f'Duplicatas encontradas: {duplicatas}')
if duplicatas > 0:
    df = df.drop_duplicates()

print(f'Base final: {df.shape[0]} rows')

## 6. Feature Engineering (básica)

Criar métricas derivadas para análise.

In [ ]:
df['Valor_Total'] = df['Preco_Unitario'] * df['Qtd']
df['Ano_Mes'] = df['Data'].dt.to_period('M').astype(str)  # para agrupamentos temporais

## 7. Integração (Join)

Merge com cadastro de gerentes para enriquecer a base.

In [ ]:
df_final = df.merge(
    df_gerentes,
    on='Loja',
    how='left',
    indicator=True  # flag para auditoria de match
)

# Auditoria: todas as lojas bateram?
print(df_final['_merge'].value_counts())

## 8. Checkpoint

Exportar base limpa para próximas etapas (análise ou modelagem).

In [ ]:
df_final.to_parquet('vendas_clean.parquet', index=False)
# parquet preserva tipos (datetime, category) melhor que csv